# Detecting inherent linearity in transformer models

This Notebook serves to experiment with detecting inherent linearity in Llama models. These experiments will be run on Llama-2-7B to test functionality and provide evidence for a feasibility study as part of the graduation preparation phase. Larger experiments for the final paper(s) will be handled using Python files in this repository.

In [1]:
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import login
from datasets import load_dataset
import re
from tqdm import tqdm
import transformers
import os

# Log in to Hugging Face with hf.login file to access the model
login(token=open("../hf.login").read().strip())

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(torch.backends.cuda.matmul.allow_tf32)
print(torch.backends.cudnn.allow_tf32)

CUDA Available: True
GPU: NVIDIA GeForce RTX 4080 SUPER
True
True


In [2]:
# Load the Llama-2-7B model and tokenizer
model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer.pad_token = tokenizer.eos_token # Set pad token to eos token if not already set to prevent errors
print("Model and tokenizer loaded.")

Model and tokenizer loaded.


In [3]:
# Load TinyStories dataset

def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", "", text)  # Remove special characters
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

# Tokenize and clean the dataset
def preprocess(examples):
    examples["text"] = [clean_text(t) for t in examples["text"]]
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)


def load_datasets():
    if not os.path.exists("./data"):
        os.makedirs("./data")

    if os.path.exists("./data/tiny_stories_train") and os.path.exists("./data/tiny_stories_val"):
        train_set = load_dataset("roneneldan/TinyStories", split="train").load_from_disk("./data/tiny_stories_train")
        val_set = load_dataset("roneneldan/TinyStories", split="validation").load_from_disk("./data/tiny_stories_val")
        print("Datasets loaded from disk.")
        return train_set, val_set

    train_set = load_dataset("roneneldan/TinyStories", split="train")
    val_set = load_dataset("roneneldan/TinyStories", split="validation")

    train_set = train_set.map(preprocess, batched=True)
    val_set = val_set.map(preprocess, batched=True)
    print("Datasets loaded and preprocessed.")

    # Save to disk for faster loading later
    train_set.save_to_disk("./data/tiny_stories_train")
    val_set.save_to_disk("./data/tiny_stories_val")
    print("Datasets saved to disk.")

    return train_set, val_set

train_dataset, val_dataset = load_datasets()

Datasets loaded from disk.


In [4]:
# Do a forward pass to ensure everything is working
def forward_pass(model, tokenizer, dataset, device='cuda', debug=False):
    model.to(device)
    model.eval()
    if debug:
        print("Preparing inputs for forward pass...")
    inputs = tokenizer(dataset[0]['text'], return_tensors='pt', truncation=True, padding='max_length', max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if debug:
        print("Inputs prepared. Performing forward pass...")
    with torch.no_grad():
        outputs = model(**inputs)
    if debug:
        print("Forward pass successful.")
    return outputs

# outputs = forward_pass(model, tokenizer, val_dataset, debug=True)
# print("Forward pass output snippet:", outputs.logits[0, :5, :5])

## Visualizing the architecture of Llama-2-7B
To aid with understanding the model architecture, we can visualize the layers and components of Llama-2-7B using a simple diagram.

In [5]:
def visualize_architecture(model):
    for name, module in model.named_modules():
        print(f"{name}: {module.__class__}")

visualize_architecture(model)

: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
model: <class 'transformers.models.llama.modeling_llama.LlamaModel'>
model.embed_tokens: <class 'torch.nn.modules.sparse.Embedding'>
model.layers: <class 'torch.nn.modules.container.ModuleList'>
model.layers.0: <class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>
model.layers.0.self_attn: <class 'transformers.models.llama.modeling_llama.LlamaAttention'>
model.layers.0.self_attn.q_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.k_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.v_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.o_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp: <class 'transformers.models.llama.modeling_llama.LlamaMLP'>
model.layers.0.mlp.gate_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.up_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.down_proj: <class 'torc

## Computing the mean of preactivations per ReLU layer
"**Mean of preactivations $\bar{p}^l$** We denote the distribution of inputs $z$ to a nonlinear activation function $f(z)$ as the preactivations. For each activation function/node, we compute the mean of the preactivations, and then we compute another mean of these values per layer $l$: $\bar{p}^l= \frac{1}{M}\sum_{i=1}^M \left(\frac{1}{N}\sum_{s=1}^N z_{s,i}^l\right)$,
with $M$ the number of nodes in layer $l$ and $N$ the number of samples, and $z_{s,i}^l$ the preactivation value for sample $s$ at node $i$ at layer $l$. We compute this value through randomly selecting 500 samples of the input data instead of the whole dataset, which significantly reduces the computational cost." (Pinson et al., 2024, p. 3).

In case there is BatchNormalization applied between the convolution and the activation function, the mean of the preactivations will be approximately 0, due to the normalization. However, BN has two learned parameters per channel, namely a scaling and a shifting parameter. The shifting parameter can be used to recover the mean of the preactivations before BN. Therefore, in case of BN, we will use the shifting parameter as the mean of the preactivations. (Pinson et al., 2024)

In [6]:
def retrieve_mean_preactivations(model, tokenizer, dataset, device='cuda'):
    """Compute the mean of preactivations for each activation layer in the model. Function does this by computing the mean preactivation values over a set of input data. For Llama with RMS normalization before and after self-attention, we can't retrieve preactivations from normalization parameters. Thus, we must calculate the mean of the input of the normalization before activation named 'model.layers.n.post_attention_layernorm' and 'model.layers.n.mlp.act_fn' where n is the layer number.
    Args:
        model: The neural network model.
        tokenizer: The tokenizer for the model.
        dataset: The dataset to compute preactivations on.
        device: Device to run the computations on.

    Returns:
        dict: A dictionary with layer names as keys and mean preactivation values as values.
    """
    model.to(device)
    model.eval()
    mean_preactivations = {}
    activation_layers = []
    hooks = []

    print("Identifying activation layers and registering hooks...")
    # Identify activation layers
    for name, module in model.named_modules():
        if re.match(r'model\.layers\.\d+\.mlp\.act_fn', name) or re.match(r'model\.layers\.\d+\.post_attention_layernorm', name):
            activation_layers.append((name, module))
    print("Identified layers, setting hooks...")
    # Define hook to capture preactivations
    def get_preactivation_hook(name):
        def hook(module, input, output):
            if name not in mean_preactivations:
                mean_preactivations[name] = []
            mean_preactivations[name].append(input[0].detach().cpu())
        return hook
    # Register hooks
    for name, module in activation_layers:
        hooks.append(module.register_forward_hook(get_preactivation_hook(name)))
    print("Hooks registered. Performing forward passes...")
    # Forward pass through the data
    with torch.no_grad():
        for i in tqdm(range(2), desc="Processing samples for preactivations", leave=False):
            inputs = tokenizer(dataset[i]['text'], return_tensors='pt', truncation=True, padding='max_length', max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            model(**inputs)
    print("Forward passes complete. Computing mean preactivations...")
    # Compute mean preactivations
    for name in mean_preactivations:
        all_preacts = torch.cat(mean_preactivations[name], dim=0)
        mean_value = all_preacts.mean().item()
        mean_preactivations[name] = mean_value
    print("Mean preactivations computed.")
    # Remove hooks
    for hook in hooks:
        hook.remove()
    print("Hooks removed.")
    return mean_preactivations


mean_preactivations = retrieve_mean_preactivations(model, tokenizer, val_dataset)
print("Mean preactivations per layer:")
for layer, mean_val in mean_preactivations.items():
    print(f"{layer}: {mean_val}")

In [7]:
# Map the preactivations of the layernorm to the preceding attention layer for clarity
mapped_mean_preactivations = {}
for layer_name, mean_val in mean_preactivations.items():
    match = re.match(r'model\.layers\.(\d+)\.post_attention_layernorm', layer_name)
    if match:
        layer_num = match.group(1)
        mapped_mean_preactivations[f'model.layers.{layer_num}.self_attn'] = mean_val
        print(f"Mapped model.layers.{layer_num}.self_attn with mean preactivation {mean_val}")

Mapped model.layers.0.self_attn with mean preactivation 0.0002632421092130244
Mapped model.layers.1.self_attn with mean preactivation 0.00037288593011908233
Mapped model.layers.2.self_attn with mean preactivation 0.0005880641983821988
Mapped model.layers.3.self_attn with mean preactivation 0.0007293073576875031
Mapped model.layers.4.self_attn with mean preactivation 0.0010481951758265495
Mapped model.layers.5.self_attn with mean preactivation 9.939318988472223e-05
Mapped model.layers.6.self_attn with mean preactivation 0.0009107987862080336
Mapped model.layers.7.self_attn with mean preactivation 0.000526921299751848
Mapped model.layers.8.self_attn with mean preactivation 0.0004824948264285922
Mapped model.layers.9.self_attn with mean preactivation 0.0007732927333563566
Mapped model.layers.10.self_attn with mean preactivation 0.0011848373105749488
Mapped model.layers.11.self_attn with mean preactivation 0.0013438827591016889
Mapped model.layers.12.self_attn with mean preactivation 0.001

# Utilizing linear approximations based on mean preactivations

To utilize inherent linearity for compression, it will be attempted to approximate the inherently linear sections of a model, based on a thresholded mean preactivation value. This approximation will be a linear layer, that must be trained to mimic the behavior of the a (section of) layer(s) it is approximating. The threshold will be determined based on the distribution of mean preactivation values across layers.

In [8]:
def choose_threshold(mean_preactivations, percentile=25):
    """Choose a threshold based on the given percentile of mean preactivation values."""
    values = list(mean_preactivations.values())
    threshold = np.percentile(values, percentile)
    return threshold

def identify_linear_layers(mean_preactivations, threshold):
    """Identify layers with mean preactivation below the threshold."""
    linear_layers = [layer for layer, mean_val in mean_preactivations.items() if mean_val < threshold]
    return linear_layers

threshold = choose_threshold(mapped_mean_preactivations, percentile=25)
linear_layers = identify_linear_layers(mapped_mean_preactivations, threshold)

In [9]:
def create_linear_approximations(model, linear_layers, device='cuda'):
    """Create linear approximations for the identified linear layers."""
    model.to(device)
    model.eval()
    linear_approximations = {}

    for layer_name in linear_layers:
        # Extract layer number
        match = re.match(r'model\.layers\.(\d+)\.self_attn', layer_name)
        if match:
            layer_num = int(match.group(1))
            # Get the original attention layer
            original_layer = model.model.layers[layer_num].self_attn
            # Create a linear layer with the same input and output dimensions
            input_dim = original_layer.q_proj.in_features
            output_dim = original_layer.q_proj.out_features
            linear_layer = torch.nn.Linear(input_dim, output_dim).to(device)
            linear_approximations[layer_name] = linear_layer
            print(f"Created linear approximation for {layer_name} with input dim {input_dim} and output dim {output_dim}")

    return linear_approximations

def train_linear_approximations(model, linear_approximations, tokenizer, dataset, device='cuda', epochs=1, lr=1e-3):
    """Train the linear approximations to mimic the behavior of the original layers."""
    model.to(device)
    model.eval()

    for layer_name, linear_layer in linear_approximations.items():
        # Extract layer number
        match = re.match(r'model\.layers\.(\d+)\.self_attn', layer_name)
        if match:
            layer_num = int(match.group(1))
            original_layer = model.model.layers[layer_num].self_attn

            optimizer = torch.optim.Adam(linear_layer.parameters(), lr=lr)
            loss_fn = torch.nn.MSELoss()

            print(f"Training linear approximation for {layer_name}...")
            for epoch in range(epochs):
                total_loss = 0.0
                for i in tqdm(range(len(dataset)), desc=f"Epoch {epoch+1}/{epochs}", leave=False):
                    inputs = tokenizer(dataset[i]['text'], return_tensors='pt', truncation=True, padding='max_length', max_length=512)
                    inputs = {k: v.to(device) for k, v in inputs.items()}

                    attention_mask = inputs["attention_mask"]
                    if attention_mask is None:
                        attention_mask = (inputs["input_ids"] != tokenizer.pad_token_id).long()

                    with torch.no_grad():
                        original_output = original_layer(
                            inputs['input_ids'],
                            attention_mask
                        )[0]

                    optimizer.zero_grad()
                    linear_output = linear_layer(inputs['input_ids'].float())
                    loss = loss_fn(linear_output, original_output)
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()
                avg_loss = total_loss / len(dataset)
                print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    print("Training complete.")

def replace_with_linear_approximations(model, linear_approximations):
    """Replace the identified linear layers in the model with their linear approximations."""
    for layer_name, linear_layer in linear_approximations.items():
        match = re.match(r'model\.layers\.(\d+)\.self_attn', layer_name)
        if match:
            layer_num = int(match.group(1))
            model.model.layers[layer_num].self_attn = linear_layer
            print(f"Replaced {layer_name} with its linear approximation.")
    return model

# Remove the original model from CUDA memory to free up space
model = model.cpu()
torch.cuda.empty_cache()

# Create, train, and replace linear approximations in a copy of the model
pruning_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
linear_approximations = create_linear_approximations(pruning_model, linear_layers)
train_linear_approximations(pruning_model, linear_approximations, tokenizer, val_dataset, device='cuda', epochs=1, lr=1e-3)
pruning_model = replace_with_linear_approximations(pruning_model, linear_approximations)

TypeError: LlamaAttention.forward() missing 1 required positional argument: 'attention_mask'